In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git

In [ ]:
# ragas 0.2.15 imports a module that langchain-community 0.4+ removed, so the LangChain family stays below 1.0
!pip install -q "transformers>=4.51" sentence-transformers qdrant-client mlflow pypdfium2 bitsandbytes "ragas==0.2.15" "langchain-community<0.4" "langchain-core<1" "langchain-text-splitters<1" "langchain-huggingface<1"

In [ ]:
import gc
import json
import os
import sys

from kaggle_secrets import UserSecretsClient

DAGSHUB_USER = "cad0m"
secrets = UserSecretsClient()

os.environ["JARIDA_PDF_DIR"] = "/kaggle/input/datasets/ayouberraouy/jarida"
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/jarida-rag.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation; must be set before torch loads
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import torch
from jarida_rag import config, evaluate, judge, rag, store

mlflow.set_tracking_uri(config.MLFLOW_URI)
# DagsHub answers a bare 404 for unknown experiment names, so look it up with search instead
found = mlflow.search_experiments(filter_string="name = 'gazette-rag'")
experiment_id = found[0].experiment_id if found else mlflow.create_experiment("gazette-rag")
mlflow.set_experiment(experiment_id=experiment_id)


def show_gpus():
    for gpu in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(gpu)
        print(f"GPU {gpu}: {free / 1024**3:.1f} GB free of {total / 1024**3:.1f} GB")


show_gpus()

In [ ]:
client = store.open_store()
embedder = store.load_embedder()
reranker = store.load_reranker()
print(store.build_index(client, embedder), "chunks")
show_gpus()

In [ ]:
evaluate.retrieval_eval(client, embedder, "retrieval-baseline")
evaluate.retrieval_eval(client, embedder, "retrieval-reranker", reranker)

In [ ]:
tokenizer, model = rag.load_llm()
print(f"LLM size: {model.get_memory_footprint() / 1024**3:.1f} GB")
show_gpus()

In [ ]:
for question in ["Which decree regulates marine aquaculture farms?", "What is the legal age of marriage in Morocco?"]:
    reply, hits = rag.answer(question, client, embedder, tokenizer, model, reranker)
    print(question)
    print(reply)
    for hit in hits:
        print("  -", rag.source(hit), hit["title"][:60])
    print()

In [ ]:
answers_baseline = evaluate.answer_questions(client, embedder, tokenizer, model)
answers_reranker = evaluate.answer_questions(client, embedder, tokenizer, model, reranker)
json.dump({"baseline": answers_baseline, "reranker": answers_reranker}, open("/kaggle/working/answers.json", "w"), ensure_ascii=False)
print(len(answers_baseline), "answers per run")

In [ ]:
# Judge phase: the answers are saved, so free both GPUs completely (both should show ~14.5 GB free)
del model, tokenizer, embedder, reranker
gc.collect()
torch.cuda.empty_cache()
show_gpus()

In [ ]:
# After a session restart, skip the cells above (except setup) and load the saved answers instead
saved = json.load(open("/kaggle/working/answers.json"))
answers_baseline, answers_reranker = saved["baseline"], saved["reranker"]

In [ ]:
judge_llm, judge_embeddings = judge.load_judge()
show_gpus()  # expected: the judge on GPU 0, the embeddings on GPU 1
# look at the raw output first: a thinking block or extra text here means RAGAS will fail to parse it
print(judge_llm.invoke('Reply with exactly this JSON and nothing else: {"ok": true}').content)

In [ ]:
# 2 questions first: if the scores are NaN, read the log above it before running the full evaluation
smoke = evaluate.ragas_eval(answers_baseline[:2], judge_llm, judge_embeddings, "ragas-smoke", reranker_used=False)
smoke

In [ ]:
baseline = evaluate.ragas_eval(answers_baseline, judge_llm, judge_embeddings, "ragas-baseline", reranker_used=False)
with_reranker = evaluate.ragas_eval(answers_reranker, judge_llm, judge_embeddings, "ragas-reranker", reranker_used=True)
baseline.mean(numeric_only=True), with_reranker.mean(numeric_only=True)

In [ ]:
# Only after pushing new code: pull it and reload the modules without restarting
!git -C /kaggle/working/jarida-rag pull

import importlib
from jarida_rag import chunk, pdf

for module in [config, pdf, chunk, store, rag, judge, evaluate]:
    importlib.reload(module)

# objects built before the reload (judge_llm, client, ...) still run the old code: re-run the cells that create them